# ECIS — Ollama on Colab

Unzip the repo, pull **Llama 3.1 8B**, **Mistral 7B**, and **Qwen2.5 14B**, then extract.

`--model llama` | `--model mistral` | `--model qwen` | `--model both` (Llama+Mistral) | `--model all` (all three).

Runtime → Change runtime type → GPU (A100 preferred for Qwen).

## Install and start Ollama:

In [ ]:
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
print("Ollama installed.")

In [ ]:
import subprocess, time

proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)
print(f"Ollama server started (PID: {proc.pid})")

In [ ]:
!ollama pull llama3.1:8b-instruct-q8_0
!ollama pull mistral:7b-instruct
!ollama pull qwen2.5:14b-instruct-q4_K_M
print("Llama 3.1 8B, Mistral 7B, and Qwen2.5 14B are ready.")

In [ ]:
import requests

resp = requests.get("http://localhost:11434/api/tags", timeout=10)
models = [m["name"] for m in resp.json().get("models", [])]
print(f"Available models: {models}")

have_llama = any("llama3.1" in m for m in models)
have_mistral = any("mistral" in m for m in models)
have_qwen = any("qwen" in m for m in models)
print(f"Llama: {have_llama}  Mistral: {have_mistral}  Qwen: {have_qwen}")
if not (have_llama and have_mistral and have_qwen):
    raise RuntimeError(
        "Need llama3.1:8b-instruct-q8_0, mistral:7b-instruct, and qwen2.5:14b-instruct-q4_K_M"
    )

In [ ]:
import os, shutil
from pathlib import Path

WORK_DIR = "/content/Mycroft_Contribution"
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
!unzip -q -o Mycroft_Contribution.zip -d /content/

os.chdir(WORK_DIR)
os.environ["PYTHONPATH"] = str(Path(WORK_DIR) / "src")
PROJECT_PATH = str(Path(WORK_DIR) / "src" / "ecis")
print(f"Working directory: {os.getcwd()}")

## Install dependencies:

In [ ]:
!pip install -q -r {PROJECT_PATH}/notebooks/colab/requirements-colab.txt
!python -m spacy download en_core_web_sm 2>/dev/null
print("Dependencies installed.")

## Configure environment:

In [ ]:
import os, re, shutil
from pathlib import Path

os.environ["ANONYMIZED_TELEMETRY"] = "False"
env_path = Path(PROJECT_PATH) / ".env"
example = Path(PROJECT_PATH) / ".env.example"
if not env_path.exists():
    if not example.exists():
        raise FileNotFoundError("No .env or .env.example found")
    shutil.copy(example, env_path)

content = env_path.read_text()
replacements = {
    r"OLLAMA_BASE_URL=.*": 'OLLAMA_BASE_URL="http://localhost:11434"',
    r"LLM_MODEL=.*": 'LLM_MODEL="llama3.1:8b-instruct-q8_0"',
    r"LLM_LLAMA_MODEL=.*": 'LLM_LLAMA_MODEL="llama3.1:8b-instruct-q8_0"',
    r"LLM_MISTRAL_MODEL=.*": 'LLM_MISTRAL_MODEL="mistral:7b-instruct"',
    r"LLM_QWEN_MODEL=.*": 'LLM_QWEN_MODEL="qwen2.5:14b-instruct-q4_K_M"',
}
for pattern, value in replacements.items():
    if re.search(pattern, content):
        content = re.sub(pattern, value, content)
    else:
        content = content.rstrip() + "\n" + value + "\n"
env_path.write_text(content)

## Initialise databases:

In [ ]:
!cd "{WORK_DIR}" && PYTHONPATH=src python -m ecis.main --init-db

## Run extraction:

In [ ]:
TICKER = "TICKER"
!cd "{WORK_DIR}" && PYTHONPATH=src python -m ecis.main --extract --ticker {TICKER} --model llama
!cd "{WORK_DIR}" && PYTHONPATH=src python -m ecis.main --extract --ticker {TICKER} --model mistral
!cd "{WORK_DIR}" && PYTHONPATH=src python -m ecis.main --extract --ticker {TICKER} --model qwen

In [ ]:
EXTRACT_MODEL = "all"  # llama | mistral | qwen | both | all
TICKERS = "TICKER"
!cd "{WORK_DIR}" && PYTHONPATH=src python -m ecis.main --extract --ticker {TICKERS} --model {EXTRACT_MODEL}

## Check results: 

In [ ]:
import sqlite3
from pathlib import Path

db_path = Path(PROJECT_PATH) / "data" / "db" / "signals.db"
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row
total = conn.execute("SELECT COUNT(*) as n FROM signals").fetchone()["n"]
print(f"Total signals: {total}")
for r in conn.execute("SELECT ticker, COUNT(*) as n FROM signals GROUP BY ticker ORDER BY n DESC"):
    print(f"  ticker {r['ticker']}: {r['n']}")
cols = {row[1] for row in conn.execute("PRAGMA table_info(signals)").fetchall()}
if "llm_model" in cols:
    print("By LLM:")
    for r in conn.execute(
        "SELECT COALESCE(llm_model, 'unknown') as model, COUNT(*) as n FROM signals GROUP BY model ORDER BY n DESC"
    ):
        print(f"  {r['model']}: {r['n']}")
conn.close()